# 0 Model names with Claude
The raw `model` column is a mess: `f150 xlt`, `f-150 supercrew` and `ford f 150` are all the same truck. Claude maps every common (manufacturer, model) name to one clean base model.
- Claude only sees the manufacturer and the raw model text and only returns the clean model; the brand is never changed
- output: `data/fixed_models.csv`, one clean model per row of `data/vehicles.csv` (indexed by the original row number), used in `data_cleaning.ipynb` (## Model)
- progress is saved after every batch, so re-running continues where it stopped; once everything is mapped it makes no API calls
- needs an Anthropic API key (`ANTHROPIC_API_KEY`)

# 1 loading dependencies and config
- `LIMIT` = only map the first N names (trial run, no output files written)

In [ ]:
import json
from pathlib import Path

import anthropic
import pandas as pd
from pydantic import BaseModel

MODEL = "claude-sonnet-5"
EFFORT = "low"
BATCH_SIZE = 100
LIMIT = None

INPUT_CSV = "data/model_map_input.csv"
PROGRESS_FILE = Path("data/model_map_progress.jsonl")
MAP_CSV = "data/model_map.csv"
FIXED_CSV = "data/fixed_models.csv"


## Input names
- `data/model_map_input.csv`: every (manufacturer, raw model) pair seen 3+ times (9,763 names), with how many listings use it
- built once from an earlier version of the cleaning and kept fixed: the progress file points into it by row number, so rebuilding it would mix up the saved mappings

In [2]:
inp = pd.read_csv(INPUT_CSV)
if LIMIT:
    inp = inp.head(LIMIT)
inp.shape

(9763, 3)

## Prompt
- lowercase base model only; the same model always gets exactly the same string
- keep the truck class (1500 / 2500 / 3500) and collapse to the model family (`328i xdrive` → `3 series`)
- junk → `unknown`, no guessing
- the answer is forced into a JSON schema: a list of (id, clean model)

In [3]:
SYSTEM_PROMPT = """You normalize messy used-car model names from Craigslist listings.

Each input line is: id|manufacturer|raw model text
Return, for every id, the real base model name as the manufacturer markets it.

Rules:
- Lowercase. Base model only: drop trim, body style, drivetrain, engine, cab type, year and the brand name.
  "mdx sh-awd sport utility" -> "mdx", "camry le" -> "camry", "civic ex-l sedan" -> "civic"
- Same model always gets exactly the same string: "f150 xlt", "f-150 supercrew", "ford f 150" -> "f-150"
- Keep the truck class 1500/2500/3500 (merge "hd" into it): "silverado 2500hd lt" -> "silverado 2500",
  ram "1500 big horn" -> "1500". Use the manufacturer to resolve bare numbers:
  chevrolet "1500" -> "silverado 1500", gmc "1500" -> "sierra 1500".
- Keep multi-word names whole: "grand cherokee", "grand caravan", "santa fe", "town and country", "model 3".
- Collapse to the model family, not the variant: bmw "328i xdrive" -> "3 series", "x5 xdrive35i" -> "x5",
  mercedes-benz "c300 4matic" / "c63 amg" -> "c-class", audi "a4 2.0t premium" -> "a4",
  lexus "rx 350" -> "rx", mazda "3 i touring" -> "mazda3".
- The manufacturer is context only. Never output a brand and never correct it.
- If the text is junk or you cannot identify a real model ("2017", "a", "truck", "all"), return "unknown". Do not guess.
"""

OUTPUT_SCHEMA = {
    "type": "object",
    "properties": {
        "mappings": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer"},
                    "model_clean": {"type": "string"},
                },
                "required": ["id", "model_clean"],
                "additionalProperties": False,
            },
        }
    },
    "required": ["mappings"],
    "additionalProperties": False,
}

class Mapping(BaseModel):
    id: int
    model_clean: str

class BatchResult(BaseModel):
    mappings: list[Mapping]

## Calling Claude
- one request per batch of 100 names
- refused or cut off → split the batch in half and try the halves
- ids Claude skipped get one retry in the run below, then `unknown`

In [4]:
def load_progress():
    if not PROGRESS_FILE.exists():
        return {}
    with PROGRESS_FILE.open() as f:
        return {r["id"]: r["model_clean"] for r in map(json.loads, f)}

def map_batch(client, rows, usage):
    """rows: list of (id, manufacturer, model_raw). Returns id -> model_clean."""
    prompt = "\n".join(f"{i}|{manufacturer}|{raw}" for i, manufacturer, raw in rows)
    response = client.messages.create(
        model=MODEL,
        max_tokens=16000,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": prompt}],
        output_config={
            "effort": EFFORT,
            "format": {"type": "json_schema", "schema": OUTPUT_SCHEMA},
        },
    )
    usage["input"] += response.usage.input_tokens
    usage["output"] += response.usage.output_tokens

    # refused or cut off: split the batch and try the halves
    if response.stop_reason in ("refusal", "max_tokens"):
        if len(rows) == 1:
            return {rows[0][0]: "unknown"}
        half = len(rows) // 2
        return map_batch(client, rows[:half], usage) | map_batch(client, rows[half:], usage)

    text = next(b.text for b in response.content if b.type == "text")
    ids = {i for i, _, _ in rows}
    return {
        m.id: m.model_clean.strip().lower() or "unknown"
        for m in BatchResult.model_validate_json(text).mappings
        if m.id in ids
    }

## Run
- only names that aren't in the progress file yet are sent
- every batch is appended to `data/model_map_progress.jsonl` right away, so stopping halfway loses nothing

In [5]:
done = load_progress()
todo = [(i, r.manufacturer, r.model_raw) for i, r in inp.iterrows() if i not in done]
print(f"{len(inp) - len(todo)} already done, {len(todo)} to map")

usage = {"input": 0, "output": 0}
if todo:
    client = anthropic.Anthropic()
    for start in range(0, len(todo), BATCH_SIZE):
        batch = todo[start : start + BATCH_SIZE]
        got = map_batch(client, batch, usage)
        missing = [row for row in batch if row[0] not in got]
        if missing:  # one retry for ids the model skipped
            got |= map_batch(client, missing, usage)
        with PROGRESS_FILE.open("a") as f:
            for i, _, _ in batch:
                f.write(json.dumps({"id": i, "model_clean": got.get(i, "unknown")}) + "\n")
        print(f"  {min(start + BATCH_SIZE, len(todo))}/{len(todo)}")

print(f"tokens this run: {usage['input']:,} in / {usage['output']:,} out")

9763 already done, 0 to map
tokens this run: 0 in / 0 out


## Results
- how many names collapse into how many clean models, and how many listings end up `unknown`
- the 20 biggest clean models (by listings) and a random sample to eyeball

In [6]:
done = load_progress()
mm = inp[["manufacturer", "model_raw"]].copy()
mm["model_clean"] = mm.index.map(done)

weights = inp["count"]
print(f"mapped {len(mm)} names -> {mm['model_clean'].nunique()} clean models")
print(f"listings mapped to 'unknown': {weights[mm['model_clean'] == 'unknown'].sum() / weights.sum():.1%}")
weights.groupby(mm["model_clean"]).sum().nlargest(20)

mapped 9763 names -> 935 clean models
listings mapped to 'unknown': 0.9%


model_clean
f-150             13835
silverado 1500     8170
1500               8087
wrangler           7036
f-250              6323
sierra 1500        5152
tacoma             5108
civic              5101
f-350              4677
accord             4676
camry              4339
3 series           4122
escape             3789
mustang            3771
2500               3446
grand cherokee     3412
explorer           3286
altima             3274
silverado          3165
tundra             3131
Name: count, dtype: int64

In [7]:
mm.sample(min(15, len(mm)), random_state=0)

,manufacturer,model_raw,model_clean
5450,jeep,dj5,dj-5
8316,toyota,highlander sport,highlander
7587,ram,2500 big horn,2500
8770,toyota,camry xle hybrid,camry
2162,chrysler,pacifica touring-l,pacifica
1765,chevrolet,avalanche crew,avalanche
2929,ford,f-150 raptor,f-150
7126,nissan,murano platnium,murano
5110,infiniti,g sedan g37x awd,g
7613,ram,1500 limited crewcab leveled,1500


## One clean model per listing
- same manufacturer fill as `data_cleaning.ipynb` (brand from the first word of the model), so the keys match the input names
- names Claude never saw (used < 3 times) become `other`, missing models become `unknown`

In [8]:
extra_brands = {"scion", "international", "genesis", "freightliner", "isuzu", "hummer",
                "oldsmobile", "maserati", "saab", "suzuki", "smart", "plymouth",
                "kenworth", "hino", "peterbilt", "sterling", "mack", "mg", "triumph", "bentley"}
miss_spellings = {"chevorlet": "chevrolet", "volkswagon": "volkswagen", "gmc,": "gmc"}

vehicles = pd.read_csv("data/vehicles.csv", usecols=["manufacturer", "model"])
first_word = vehicles["model"].str.split().str[0].str.lower().replace(miss_spellings)
known = set(vehicles["manufacturer"].dropna().unique()) | extra_brands
manufacturer = vehicles["manufacturer"].fillna(first_word.where(first_word.isin(known))).fillna("unknown")

keys = pd.MultiIndex.from_arrays([manufacturer, vehicles["model"].str.lower().str.strip()])
lookup = mm.set_index(["manufacturer", "model_raw"])["model_clean"]
fixed = pd.Series(lookup.reindex(keys).to_numpy(), index=vehicles.index, name="model").fillna("other")
fixed[vehicles["model"].isna()] = "unknown"
fixed.value_counts().head(10)

model
other             27535
f-150             15388
silverado 1500     9216
1500               9028
unknown            8803
wrangler           7933
f-250              6902
sierra 1500        5640
civic              5567
tacoma             5512
Name: count, dtype: int64

# DF TO CSV
- skipped on a trial run (`LIMIT`); the progress is kept for the full run

In [9]:
if LIMIT:
    print("trial run: no output files written")
else:
    mm.to_csv(MAP_CSV, index=False)
    fixed.to_csv(FIXED_CSV, index_label="row")
    print(f"wrote {MAP_CSV} and {FIXED_CSV} ({len(fixed):,} rows)")

wrote data/model_map.csv and data/fixed_models.csv (426,880 rows)
